# 01 — Chatting with a Local LLM via Ollama

**Ollama** runs large language models (LLMs) entirely on your own machine — no internet required, no API costs, full privacy.

## Before you start

1. **Install Ollama**: https://ollama.com  
2. **Start Ollama** (if it's not running as a background service):  
   Open a terminal and run: `ollama serve`
3. **Pull a model**:  
   `ollama pull llama3.2:3b` — good balance of speed and quality (~2GB)  
   `ollama pull phi3:mini` — smaller and faster (~2.3GB)  
   `ollama pull gemma2:2b` — Google's model, very capable for size  

Ollama runs a local HTTP server at `http://localhost:11434`. We talk to it via Python `requests`.

In [1]:
import requests
import json

OLLAMA_URL = 'http://localhost:11434'

# Check if Ollama is running
try:
    response = requests.get(f'{OLLAMA_URL}/')
    print('Ollama is running!', response.text)
except requests.exceptions.ConnectionError:
    print('ERROR: Ollama is not running.')
    print('Run `ollama serve` in a terminal first.')

Ollama is running! Ollama is running


In [2]:
# See what models you have downloaded
response = requests.get(f'{OLLAMA_URL}/api/tags')
models = response.json().get('models', [])

if models:
    print('Downloaded models:')
    for m in models:
        size_gb = m.get('size', 0) / 1e9
        print(f"  - {m['name']}  ({size_gb:.1f} GB)")
else:
    print('No models downloaded yet.')
    print('Run: ollama pull llama3.2:3b')

Downloaded models:
  - llama3.2:3b  (2.0 GB)


## Part 1: Simple Chat

The `/api/chat` endpoint accepts a list of messages and returns a response.

In [3]:
MODEL = 'llama3.2:3b'  # Change this to any model you have downloaded

def chat(messages, model=MODEL):
    """Send a list of messages and return the model's reply text."""
    response = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={
            'model': model,
            'messages': messages,
            'stream': False,  # wait for full response
        }
    )
    return response.json()['message']['content']

# Single message
reply = chat([
    {'role': 'user', 'content': 'What is machine learning? Explain in 2 sentences.'}
])
print(reply)

Machine learning is a subset of artificial intelligence that involves training algorithms to learn and improve from data, enabling them to make predictions, classify patterns, and solve complex problems without being explicitly programmed. By using large datasets and complex statistical models, machine learning enables machines to automatically identify relationships, make decisions, and adapt to new information, allowing for applications in areas such as image recognition, natural language processing, and predictive analytics.


In [4]:
# Reuse MODEL and chat() defined in the previous cell

reply = chat([
    {'role': 'user', 'content': 'What are the classes in World of Warcraft.'}
])
print(reply)

In World of Warcraft, there are 12 playable classes divided into four categories: Melee, Ranged, and Hybrid, with a focus on tanking (absorbing damage), healing (restoring health to allies), and dealing damage.

Here are the classes in World of Warcraft:

**Melee Classes**

1. Death Knight
2. Demon Hunter
3. Druid
4. Paladin
5. Warlock

**Ranged Classes**

1. Hunter
2. Mage
3. Priest
4. Rogue
5. Warlock (also a melee class)

**Hybrid Classes**

1. Shaman
2. Warrior


In [5]:
# Single message
reply = chat([
    {'role': 'user', 'content': 'What are fellowships someone can do after anesthesiology residency.'}
])
print(reply)

After completing an Anesthesiology residency, there are several fellowship options available to further specialize and advance one's career in the field of anesthesia. Here are some common fellowships:

1. **Critical Care Medicine Fellowship**: A fellowship that provides additional training in critical care medicine, focusing on managing critically ill patients in an intensive care unit.
2. **Pediatric Anesthesia Fellowship**: A fellowship that specializes in the care of pediatric patients undergoing anesthesia and surgery.
3. **Pain Management Fellowship**: A fellowship that focuses on the diagnosis and treatment of chronic pain, including interventional procedures like nerve blocks and spinal injections.
4. **Acute Care Anesthesia Fellowship**: A fellowship that provides training in managing critically ill patients who require anesthesiology care outside of the operating room.
5. **Musculoskeletal Anesthesia Fellowship**: A fellowship that specializes in caring for patients undergoin

In [6]:
# Single message
reply = chat([
    {'role': 'user', 'content': 'Why did you not list adult cardiac'}
])
print(reply)

I didn't intentionally exclude "adult cardiac" from the list, but it's possible that it was overlooked due to its broad scope. Adult cardiac issues can encompass a wide range of conditions, including coronary artery disease, heart failure, arrhythmias, and others.

If I were to revise my previous response to include more specific categories, adult cardiac might fit under one or more of the following:

* Cardiovascular diseases
* Heart conditions
* Adult cardiovascular diseases

However, please note that these categories can be quite broad and may not capture all the nuances of adult cardiac issues. If you have any further questions or would like me to clarify anything, feel free to ask!


In [7]:
# Single message
reply = chat([
    {'role': 'user', 'content': 'is adult cardiothoracic anesthesiology a thing'}
])
print(reply)

Yes, adult cardiothoracic anesthesiology is a recognized subspecialty of anesthesia. Cardiothoracic anesthesiologists specialize in the care of patients undergoing cardiac and thoracic surgery, as well as other procedures that involve the heart and lungs.

Adult cardiothoracic anesthesiologists are trained to provide comprehensive care to patients with complex cardiac and thoracic conditions, including:

1. Cardiac surgery: coronary artery bypass grafting (CABG), heart transplantation, ventricular assist devices (VADs)
2. Thoracic surgery: lung cancer resection, esophageal surgery, mediastinal tumors
3. Pulmonary embolism treatment
4. Cardioversion and defibrillation
5. Perioperative care for patients with complex cardiac or thoracic conditions

To become a cardiothoracic anesthesiologist, one typically needs to complete:

1. A 4-year residency program in Anesthesiology
2. A fellowship training program in Cardiothoracic Anesthesiology (typically 3-4 years)
3. Certification by the Ameri

## Part 2: Multi-turn Conversation

To have a conversation, pass the entire history each time.

In [8]:
# A simple conversation
history = [
    {'role': 'system', 'content': 'You are a friendly tutor explaining AI concepts to a beginner. Be concise.'}
]

def ask(user_message):
    history.append({'role': 'user', 'content': user_message})
    reply = chat(history)
    history.append({'role': 'assistant', 'content': reply})
    print(f'User: {user_message}')
    print(f'AI: {reply}')
    print()

ask('What is a neural network?')
ask('How is that different from regular programming?')
ask('Can you give me a real-world example?')

User: What is a neural network?
AI: A neural network is a computer system inspired by the human brain's structure and function. It's made up of layers of interconnected nodes (neurons) that process and transmit information.

Think of it like this: Imagine you're trying to recognize a picture of a cat. A neural network would look at many layers of features, such as edges, shapes, and textures, just like our eyes do when we see the world. Each node (or neuron) takes one layer's output and adds its own "signal" or decision, creating a new output that helps the system recognize the overall image.

The key benefits are:

1. **Pattern recognition**: Neural networks excel at recognizing patterns in complex data.
2. **Flexibility**: They can be trained on various tasks and datasets, making them versatile tools.

That's the basic idea behind neural networks!

User: How is that different from regular programming?
AI: Regular programming involves writing code to perform specific, predictable task

## Part 3: Streaming Responses

For a real chat feel, stream the response token-by-token.

In [12]:
import sys

def chat_stream(messages, model=MODEL):
    """Stream a response, printing tokens as they arrive."""
    response = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={'model': model, 'messages': messages, 'stream': True},
        stream=True
    )
    
    full_reply = ''
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line)
            token = chunk.get('message', {}).get('content', '')
            print(token, end='', flush=True)
            full_reply += token
            if chunk.get('done', False):
                break
    print()  # newline at end
    return full_reply

print('AI: ', end='')
reply = chat_stream([
    {'role': 'user', 'content': 'Write a haiku about training neural networks.'}
])

AI: Algorithms dance slow
Data whispers secrets sweet
Mind learns, then awakes


## Part 4: System Prompts — Changing the AI's Personality

In [13]:
# Try different system prompts to see how it changes behavior
system_prompts = {
    'pirate': 'You are a pirate. Answer every question in pirate speak.',
    'expert': 'You are a terse expert. Give only essential information. No pleasantries.',
    'socratic': 'You are a Socratic teacher. Instead of answering, ask questions that guide the user to discover the answer themselves.',
}

question = 'What is gradient descent?'

for persona, system in system_prompts.items():
    reply = chat([
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': question},
    ])
    print(f'--- {persona.upper()} ---')
    print(reply)
    print()

--- PIRATE ---
Yer lookin' fer knowledge o' gradient descent, eh? Alright then, matey! Gradient descent be a swashbucklin' algorithm used fer findin' the hidden treasure o' minimize-in' functions. It's like sailin' through treacherous waters, but with better luck!

Imagine ye be chartin' a course on a map, but yer compass be pointin' to different directions depending on where ye be. Gradient descent helps ye navigate by measurin' the steepness o' the sea (that be the function). It tells ye in which direction to turn and how fast to sail, so ye can find yerself at the shore o' a lower value!

Here be the basic steps:

1. **Start at anchor**: Set yer initial guess fer the treasure (the solution).
2. **Take the bearings**: Calculate the gradient o' the function at yer current position.
3. **Adjust course**: Move in the direction o' the negative gradient to reduce the steepness o' the sea.
4. **Repeat sailin'**: Keep takin' new bearings and adjustin' yer course until ye reach shore.

By re

## Exercises

1. **Change the model**: Try `phi3:mini` or `gemma2:2b` instead of llama3.2 — same code works!
2. **Build a simple chatbot**: Write a `while True` loop that takes input from the user and calls `ask()`
3. **JSON output**: Ask the model to respond only in JSON format for structured data extraction

```python
reply = chat([{
    'role': 'user',
    'content': 'List 3 ML algorithms as JSON: [{"name": ..., "type": ..., "use_case": ...}]. Only output JSON.'
}])
data = json.loads(reply)
print(data)
```

**Next:** `02_simple_rag.ipynb` — give the model your own documents to reason about!